# **Modelos Predictivos Saber 11 - Departamento de Caldas**

Este notebook implementa el proceso de selección, limpieza, alistamiento y análisis exploratorio de los datos de las pruebas Saber 11 para el departamento de Caldas, como parte del Proyecto 2 del curso *Analítica Computacional para la Toma de Decisiones*. El producto final está orientado al **Ministerio de Educación** como usuario final, y busca responder tres preguntas de negocio relacionadas con equidad socioeconómica, desempeño territorial y brechas de género.

## Tarea 4 - Modelamiento

Se exploran diferentes configuraciones de modelo, se realiza ingeniería de características, se emplean diferentes métodos de estimación, y se comparan y seleccionan las mejores alternativas.

---

Daniel Benavides - 202220428 

Juanita Cortés - 202222129 

Andrés Felipe Herrera - 202220888

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
from prettytable import PrettyTable

from matplotlib import font_manager
plt.rcParams['font.family'] = 'Arial'

alt.data_transformers.enable('vegafusion')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, callbacks

import mlflow
mlflow.set_tracking_uri("mlruns")

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Carga de datos

In [ ]:
df = pd.read_csv('../data/saber11_features.csv')

print(f"Filas: {len(df)} | Columnas: {df.shape[1]}")
df.head(5)

## 1. Pregunta de negocio

### *¿Cuál es el puntaje global esperado para un estudiante de Caldas dado su perfil socioeconómico y el tipo de institución educativa?*

Esta pregunta se responde con un **modelo de regresión**, ya que la variable objetivo es continua:

$$
Y = punt\_global
$$

El objetivo es estimar el puntaje global esperado de un estudiante del departamento de Caldas a partir de variables asociadas a su contexto socioeconómico y a las características de la institución educativa. Esta predicción puede apoyar la identificación de perfiles con menor desempeño esperado y orientar estrategias de acompañamiento académico.

### 1.1 Feature Engineering

In [ ]:
df_p1 = df.dropna(subset=['punt_global']).copy()
TARGET_P1 = 'punt_global'

FEATURES_P1 = [
    'estrato_num', 'edu_madre_num', 'edu_padre_num',
    'fami_tienecomputador', 'fami_tieneinternet',
    'es_privado', 'es_urbano'
]
df_model_p1 = df_p1[FEATURES_P1 + [TARGET_P1]].copy()

print(f"Shape: {df_model_p1.shape}")
df_model_p1.head(3)

In [ ]:
X = df_model_p1[FEATURES_P1].values
y = df_model_p1[TARGET_P1].values

X = SimpleImputer(strategy='median').fit_transform(X)
X = StandardScaler().fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val,  y_train, y_val  = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

### 1.2 Arquitectura del modelo

In [ ]:
def model_simple():
    m = tf.keras.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)
    ], name="simple")
    m.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return m

def model_deep():
    m = tf.keras.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ], name="deep_dropout")
    m.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return m

### 1.3 Entrenamiento con MLFlow

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

mlflow.set_experiment("P1_regresion_punt_global")

resultados_p1 = {}

for nombre, build_fn in [("simple", model_simple), ("deep_dropout", model_deep)]:
    with mlflow.start_run(run_name=nombre):
        model = build_fn()
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=100, batch_size=64,
            callbacks=[early_stop],
            verbose=0
        )

        y_pred = model.predict(X_test, verbose=0).flatten()
        mae  = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2   = r2_score(y_test, y_pred)

        mlflow.log_params({"arquitectura": nombre, "epochs": len(history.history['loss'])})
        mlflow.log_metrics({"mae": mae, "rmse": rmse, "r2": r2})

        model.save(f"model_{nombre}.keras")

        resultados_p1[nombre] = {"mae": mae, "rmse": rmse, "r2": r2,
                                  "history": history, "y_pred": y_pred}
        print(f"[{nombre}]  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}")

### 1.4 Comaparción de modelos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (nombre, res) in zip(axes, resultados_p1.items()):
    ax.plot(res['history'].history['loss'],     label='Train')
    ax.plot(res['history'].history['val_loss'], label='Val')
    ax.set_title(f"{nombre} — Loss (MSE)")
    ax.set_xlabel("Época")
    ax.set_ylabel("MSE")
    ax.legend()

plt.tight_layout()
plt.show()

tabla = PrettyTable()
tabla.field_names = ["Arquitectura", "MAE", "RMSE", "R²"]
for nombre, res in resultados_p1.items():
    tabla.add_row([nombre, f"{res['mae']:.2f}", f"{res['rmse']:.2f}", f"{res['r2']:.4f}"])
print(tabla)

## 2. Pregunta de negocio

### *¿Puede identificarse si un estudiante está en riesgo de obtener un puntaje global por debajo del umbral de bajo desempeño, según sus características socioeconómicas y escolares?*

Esta pregunta se responde con un **modelo de clasificación binaria**. La variable objetivo es:

$$\text{bajo\_rendimiento} = \begin{cases} 1 & \text{si } \texttt{punt\_global} < 250 \\ 0 & \text{en otro caso} \end{cases}$$

El umbral de 250 puntos es consistente con el análisis del Proyecto 1, donde se identificó ese valor como frontera entre municipios de rendimiento crítico y rendimiento medio-alto.

## 3. Pregunta de negocio

### *¿Puede predecirse el nivel de desempeño en inglés de un estudiante (A−, A1, A2, B1, B+) a partir de su perfil académico, socioeconómico y de género?*